# Baseline TabICL — Retouche
Mismo baseline y orden de trabajo que las referencias de LGBM/XGBoost.
Configuración en `config_tfm.yaml`, ubicado junto a la notebook.

- Búsqueda: una semilla y una partición 70/30; undersampling de CONTINUA en entrenamiento y validación.
- Cada combinación parte de un adaptador nuevo. El TFM permanece congelado.
- Validación por AUC en las épocas indicadas en el YAML; 202106 queda para producción.
- `Grid_log.txt` guarda los parámetros y el resultado de cada evaluación.

Ejecutar con el entorno CUDA ya utilizado y el kernel recién reiniciado.


## 1. Seteo del ambiente local


In [1]:
import os
import gc
import json
import math
import random
import tempfile
import time
from contextlib import ExitStack
from copy import deepcopy
from datetime import datetime
from itertools import product
from pathlib import Path
from collections import OrderedDict

import yaml
import numpy as np
import pandas as pd
import traceback


## 2. Definición de parámetros
### 2.1 Leemos .yaml


In [2]:
ARCHIVO_CONFIG = Path("config_tfm.yaml").resolve()
with ARCHIVO_CONFIG.open(encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

PARAM = {
    "training": deepcopy(config["training"]),
    "fijos": deepcopy(config["tabicl"]["retouche"]["fijos"]),
    "ejecucion": deepcopy(config["tabicl"]["ejecucion"]),
}
hs = config["tabicl"]["retouche"]["hs"]
SEED = int(PARAM["training"]["semilla_primigenia"][0])

DATA_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["datasets"]).resolve()
EXP_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["exp"] / config["exp_name"]).resolve()
PRED_DIR = (ARCHIVO_CONFIG.parent / config["directorios"]["predicciones"]).resolve()
assert DATA_DIR.is_dir(), DATA_DIR
EXP_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVO_LOG = EXP_DIR / "Grid_log.txt"
print("Experimento:", EXP_DIR)


Experimento: /home/marco/code/DMEyF/exp/TabICL_Retouche


In [3]:
# Configurar antes de importar PyTorch e inicializar CUDA.
ejecucion = PARAM["ejecucion"]
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ejecucion["cublas_workspace_config"]
for nombre in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[nombre] = str(ejecucion["hilos_cpu"])

import torch
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tabicl import FinetunedTabICLClassifier
from tabicl._finetune.data import MetaBatch, _split_ctx_query, move_meta_batch
from tabicl._sklearn.preprocessing import TransformToNumerical, EnsembleGenerator

DEVICE = torch.device(ejecucion["dispositivo"])
assert torch.cuda.is_available(), "Seleccionar el entorno PyTorch CUDA."
assert ejecucion["precision_evaluacion"] == ejecucion["precision_adaptador"] == "fp32"
torch.cuda.set_device(DEVICE)
torch.set_num_threads(ejecucion["hilos_cpu"])
torch.backends.cuda.matmul.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.allow_tf32 = ejecucion["permitir_tf32"]
torch.backends.cudnn.deterministic = ejecucion["cudnn_deterministic"]
torch.use_deterministic_algorithms(
    ejecucion["algoritmos_deterministas"], warn_only=ejecucion["determinismo_warn_only"]
)

def fijar_semilla(semilla):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)

fijar_semilla(SEED)
print(torch.cuda.get_device_name(DEVICE))


[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.5.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


NVIDIA GeForce RTX 3090


### 2.2 Funciones de partición y registro


In [4]:
def seleccionar_training(indices, semilla, training):
    # Mismo criterio de undersampling que LGBM/XGB: todas las bajas y 10% de CONTINUA.
    azar = np.random.default_rng(semilla).random(len(indices))
    bajas = metadatos.loc[indices, "clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).to_numpy()
    return indices[(azar <= training["undersampling"]) | bajas]


def loguear(iteracion, epoca, metrica, parametros):
    registro = {
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "iter": iteracion, "epoch": epoca, "metrica": metrica,
        "semilla": SEED, **parametros["hp"],
        # Incluye los parámetros fijos y de datos para recuperar la corrida.
        "parametros": json.dumps(parametros, ensure_ascii=False),
    }
    pd.DataFrame([registro]).to_csv(
        ARCHIVO_LOG, sep="\t", index=False, mode="a", header=not ARCHIVO_LOG.exists()
    )
    print(f"Validación | combinación={iteracion} | época={epoca} | AUC={metrica:.6f}", flush=True)


## 3. Dataset
### 3.1 Lectura y data quality


In [5]:
historia = pd.read_csv(DATA_DIR / PARAM["training"]["dataset"], low_memory=False)
historia = historia.loc[historia["foto_mes"] <= max(PARAM["training"]["future"])]
historia = historia.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
campos_lag = [c for c in historia.select_dtypes(include="number").columns
              if c not in PARAM["training"]["campos_excluidos"]]

# Variable numérica toda cero y sin NA en un mes: convertir esos valores a NA.
for campo in campos_lag:
    todo_cero = historia[campo].eq(0).groupby(historia["foto_mes"]).all()
    historia.loc[historia["foto_mes"].isin(todo_cero.index[todo_cero]), campo] = np.nan

print("Historial:", historia.shape)


Historial: (4378500, 155)


### 3.2 Feature engineering: lag1, lag2, delta1 y delta2


In [6]:
# Materializar una sola matriz CPU; conservar el baseline completo en RAM.
campos_base = [c for c in historia.columns
               if c not in {"clase_ternaria", "clase01", "fold", "azar", "training"}]
columnas = (campos_base + [f"{c}_lag1" for c in campos_lag]
            + [f"{c}_lag2" for c in campos_lag]
            + [f"{c}_delta{k}" for c in campos_lag for k in (1, 2)])
X_baseline = np.empty((len(historia), len(columnas)), dtype=np.float64, order="F")
posiciones = {c: i for i, c in enumerate(columnas)}
for campo in campos_base:
    X_baseline[:, posiciones[campo]] = historia[campo].to_numpy(dtype=np.float64)
for campo in campos_lag:
    actual = historia[campo].to_numpy(dtype=np.float64)
    grupo = historia.groupby("numero_de_cliente", sort=False)[campo]
    for k in (1, 2):
        lag = grupo.shift(k).to_numpy(dtype=np.float64)
        X_baseline[:, posiciones[f"{campo}_lag{k}"]] = lag
        X_baseline[:, posiciones[f"{campo}_delta{k}"]] = actual - lag

metadatos = historia[["numero_de_cliente", "foto_mes", "clase_ternaria"]].copy()
metadatos["clase_ternaria"] = metadatos["clase_ternaria"].astype("category")
metadatos["clase01"] = metadatos["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int64)
dataset_baseline = pd.DataFrame(X_baseline, columns=columnas, copy=False)
del historia, grupo, lag, actual
gc.collect()
print("Baseline completo en RAM:", dataset_baseline.shape)
print("Columnas para entrenar:", X_baseline.shape[1], "(referencia: 762)")


Baseline completo en RAM: (4378500, 762)
Columnas para entrenar: 762 (referencia: 762)


### 3.3 Partición 70/30 y undersampling


In [7]:
indices = metadatos.index[metadatos["foto_mes"].isin(PARAM["training"]["train"])].to_numpy()
idx_train, idx_valid = train_test_split(
    indices, train_size=PARAM["training"]["training_pct"] / 100,
    stratify=metadatos.loc[indices, "clase_ternaria"], random_state=SEED,
)
idx_train = seleccionar_training(idx_train, SEED, PARAM["training"])
# Una muestra fija del 30%: mismo factor de undersampling que en training.
idx_valid = seleccionar_training(idx_valid, SEED, PARAM["training"])
assert not np.intersect1d(idx_train, idx_valid).size
assert not set(PARAM["training"]["train"]) & set(PARAM["training"]["future"])
print("Train con undersampling:", len(idx_train))
print("Validación con undersampling:", len(idx_valid))


Train con undersampling: 306255
Validación con undersampling: 131152


## 4. Preprocesamiento de la API
Se ajusta una vez, únicamente con el training seleccionado:

1. Imputación numérica de la API.
2. Filtro de columnas constantes, escalado y recorte de extremos de la API.
3. Aplicación del mismo tratamiento a validación, sin volver a ajustarlo.

El adaptador recibe la salida de este tratamiento. No se agregan transformaciones manuales.


In [8]:
def preparar_datos(indices_train, indices_eval, semilla, parametros):
    numerico = TransformToNumerical()
    X_numerico = numerico.fit_transform(X_baseline[indices_train])
    y = metadatos.loc[indices_train, "clase01"].to_numpy()
    generador = EnsembleGenerator(
        classification=True,
        n_estimators=parametros["fijos"]["n_estimators_finetune"],
        random_state=semilla, **parametros["fijos"]["preprocesamiento"],
    )
    generador.fit(X_numerico, y)
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    X_train = np.ascontiguousarray(generador.preprocessors_[metodo].X_transformed_, dtype=np.float32)
    generador.X_ = generador.y_ = None
    generador.preprocessors_[metodo].X_transformed_ = None
    del X_numerico
    gc.collect()

    X_eval = np.empty((len(indices_eval), X_train.shape[1]), dtype=np.float32)
    bloque = parametros["ejecucion"]["tamanos_consulta_oom"][0]
    for desde in range(0, len(indices_eval), bloque):
        hasta = min(desde + bloque, len(indices_eval))
        entrada = numerico.transform(X_baseline[indices_eval[desde:hasta]])
        X_eval[desde:hasta] = generador.transform(entrada, mode="test")[metodo][0][0]
    assert np.isfinite(X_train).all() and np.isfinite(X_eval).all()
    return X_train, y, X_eval


In [9]:
X_train, y_train, X_valid = preparar_datos(idx_train, idx_valid, SEED, PARAM)
y_valid = metadatos.loc[idx_valid, "clase01"].to_numpy()
print("Training:", X_train.shape, "| validación:", X_valid.shape)


Training: (306255, 762) | validación: (131152, 762)


## 5. Modelo
### 5.1 TFM congelado y adaptador cross


In [10]:
fijos = PARAM["fijos"]
trainer = FinetunedTabICLClassifier(
    device=str(DEVICE), random_state=SEED,
    n_estimators_finetune=fijos["n_estimators_finetune"],
    n_estimators_inference=fijos["n_estimators_inference"],
    freeze_col=fijos["freeze_col"], freeze_row=fijos["freeze_row"], freeze_icl=fijos["freeze_icl"],
    **fijos["preprocesamiento"],
)
trainer._load_pretrained(DEVICE)
modelo = trainer.model_.to(DEVICE)
trainer._apply_freezing(modelo)
assert all(not p.requires_grad for p in modelo.parameters())


class CapaCross(torch.nn.Module):
    def __init__(self, dimension, rango, hp, parametros):
        super().__init__()
        self.reducir = torch.nn.Linear(dimension, rango, bias=False)
        self.expandir = torch.nn.Linear(rango, dimension)
        self.normalizar = torch.nn.BatchNorm1d(dimension) if hp["batchnorm"] else torch.nn.Identity()
        inicializar = getattr(torch.nn.init, parametros["inicializacion"] + "_")
        inicializar(self.reducir.weight, std=parametros["std_inicial"])
        inicializar(self.expandir.weight, std=parametros["std_inicial"])
        torch.nn.init.constant_(self.expandir.bias, parametros["sesgo_inicial"])

    def forward(self, x0, x):
        return x + self.normalizar(x0 * self.expandir(self.reducir(x)))


class AdaptadorCross(torch.nn.Module):
    def __init__(self, dimension, hp, parametros):
        super().__init__()
        assert parametros["tipo"] == "cross"
        rango = max(1, int(dimension * parametros["rango_relativo"]))
        self.capas = torch.nn.ModuleList([
            CapaCross(dimension, rango, hp, parametros) for _ in range(parametros["capas"])
        ])
        self.alpha = torch.nn.Parameter(torch.full((dimension,), hp["compuerta_inicial"]))

    def forward(self, X):
        # El adaptador calcula en FP32; el TFM conserva el AMP del training oficial.
        with torch.autocast(device_type=X.device.type, enabled=False):
            x0 = X.float().reshape(-1, X.shape[-1])
            x = x0
            for capa in self.capas:
                x = capa(x0, x)
            return (x0 + self.alpha * (x - x0)).reshape(X.shape)


### 5.2 Evaluación FP32 con contexto completo
Se reutiliza la caché del contexto y se procesan las consultas por bloques.
Se conservan las dos optimizaciones de memoria de la prueba anterior: compactar K/V y
no duplicar la caché de un único estimador. No cambian las predicciones.


In [11]:
# class EntradaPreparada:
#     def transform(self, X):
#         return np.asarray(X)


# class VistaPreparada:
#     def __init__(self, dimension, metodo):
#         self.metodo = metodo
#         self.feature_shuffles_ = {metodo: [list(range(dimension))]}
#         self.class_shuffles_ = {metodo: [[0, 1]]}

#     def transform(self, X, mode="test"):
#         return OrderedDict([(self.metodo, (np.asarray(X)[None],))])


# def compactar_kv(modulo, entrada, salida):
#     if isinstance(salida, tuple):
#         kv = [t.clone(memory_format=torch.contiguous_format)
#               if t.untyped_storage().nbytes() > t.numel() * t.element_size() else t
#               for t in salida[1:]]
#         torch.cuda.synchronize(DEVICE)
#         return (salida[0], *kv)
#     torch.cuda.synchronize(DEVICE)


# def preparar_cache(estimador, X, y, metodo):
#     estimador._resolve_device()
#     estimador._build_inference_config()
#     estimador.n_samples_in_, estimador.n_features_in_ = X.shape
#     estimador.y_encoder_ = LabelEncoder().fit(y)
#     estimador.classes_ = estimador.y_encoder_.classes_
#     estimador.n_classes_ = len(estimador.classes_)
#     estimador.X_encoder_ = EntradaPreparada()
#     estimador.ensemble_generator_ = VistaPreparada(X.shape[1], metodo)
#     estimador.cache_mode_ = estimador.kv_cache
#     entrada = torch.from_numpy(X[None]).float().to(DEVICE)
#     etiquetas = torch.from_numpy(y[None]).float().to(DEVICE)
#     try:
#         with ExitStack() as pila:
#             # Conservar las fronteras sincronizadas de la evaluación que funcionó.
#             pila.callback(modelo.row_interactor.register_forward_hook(
#                 lambda *args: torch.cuda.synchronize(DEVICE)).remove)
#             for bloque in modelo.icl_predictor.tf_icl.blocks:
#                 pila.callback(bloque.register_forward_pre_hook(
#                     lambda *args: torch.cuda.synchronize(DEVICE)).remove)
#                 pila.callback(bloque.register_forward_hook(compactar_kv).remove)
#             modelo.forward_with_cache(
#                 X_train=entrada, y_train=etiquetas,
#                 store_cache=True, cache_mode=estimador.cache_mode_,
#                 inference_config=estimador.inference_config_,
#             )
#     finally:
#         del entrada, etiquetas
#     estimador.model_kv_cache_ = OrderedDict([(metodo, modelo._cache)])
#     modelo.clear_cache()


In [12]:
# import traceback
# from tqdm.auto import tqdm


# def adaptar_numpy(X, adaptador):
#     if adaptador is None:
#         return X
#     with torch.no_grad():
#         entrada = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
#         salida = adaptador(entrada).float().cpu().numpy()
#     assert salida.shape == X.shape and np.isfinite(salida).all()
#     return salida


# @torch.no_grad()
# def predecir(X_contexto, y_contexto, X_consultas, adaptador, parametros):
#     e = parametros["ejecucion"]
#     metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
#     tamanos = e["tamanos_consulta_oom"]
#     modo_tfm = modelo.training
#     modo_adaptador = adaptador.training if adaptador is not None else None
#     modelo.eval()
#     if adaptador is not None:
#         adaptador.eval()
#         adaptador.zero_grad(set_to_none=True)
#     modelo.clear_cache()
#     modelo.zero_grad(set_to_none=True)
#     gc.collect()
#     torch.cuda.empty_cache()
#     contexto = estimador = None
#     raiz = Path(e["offload_raiz"]).expanduser()
#     raiz.mkdir(parents=True, exist_ok=True)
#     try:
#         with tempfile.TemporaryDirectory(dir=raiz) as temporal:
#             comun = {
#                 "use_amp": False, "use_fa3": e["use_fa3"],
#                 "safety_factor": e["safety_factor"], "use_async": e["use_async"],
#             }
#             inferencia = {
#                 nombre: dict(comun)
#                 for nombre in ("COL_CONFIG", "ROW_CONFIG", "ICL_CONFIG")
#             }
#             inferencia["COL_CONFIG"].update(
#                 cpu_safety_factor=e["cpu_safety_factor"],
#                 max_pinned_memory_mb=e["max_pinned_memory_mb"],
#                 disk_offload_dir=temporal,
#             )
#             estimador = trainer._build_inner_estimator(
#                 modelo, parametros["fijos"]["n_estimators_inference"], DEVICE
#             )
#             estimador.set_params(
#                 kv_cache=e["kv_cache"], batch_size=e["batch_size_api"],
#                 use_amp=False, use_fa3=e["use_fa3"],
#                 disk_offload_dir=temporal, inference_config=inferencia,
#             )
#             with torch.autocast(device_type=DEVICE.type, enabled=False):
#                 contexto = X_contexto
#                 if adaptador is not None:
#                     contexto = np.empty_like(X_contexto)
#                     for desde in tqdm(
#                         range(0, len(contexto), tamanos[0]),
#                         desc="Adaptando contexto", unit="bloque",
#                     ):
#                         hasta = min(desde + tamanos[0], len(contexto))
#                         contexto[desde:hasta] = adaptar_numpy(X_contexto[desde:hasta], adaptador)

#                 print(f"Creando caché del contexto: {len(contexto):,} filas...", flush=True)
#                 inicio = time.perf_counter()
#                 preparar_cache(estimador, contexto, y_contexto, metodo)
#                 torch.cuda.synchronize(DEVICE)
#                 print(f"Caché lista en {(time.perf_counter()-inicio)/60:.2f} min.", flush=True)
#                 contexto = None
#                 proba = np.empty(len(X_consultas), dtype=np.float64)
#                 desde = indice_tamano = 0
#                 with tqdm(total=len(X_consultas), desc="Prediciendo", unit="filas") as progreso:
#                     while desde < len(X_consultas):
#                         hasta = min(desde + tamanos[indice_tamano], len(X_consultas))
#                         try:
#                             entrada = adaptar_numpy(X_consultas[desde:hasta], adaptador)
#                             salida = estimador.predict_proba(entrada)
#                             torch.cuda.synchronize(DEVICE)
#                             if not np.isfinite(salida).all():
#                                 raise FloatingPointError(f"Probabilidades no finitas en filas {desde}:{hasta}.")
#                             proba[desde:hasta] = salida[:, 1]
#                         except torch.OutOfMemoryError as error:
#                             traceback.clear_frames(error.__traceback__)
#                             modelo.clear_cache()
#                             gc.collect()
#                             torch.cuda.empty_cache()
#                             indice_tamano += 1
#                             if indice_tamano == len(tamanos):
#                                 raise
#                             tqdm.write(f"OOM de consulta; reintentando con {tamanos[indice_tamano]} filas.")
#                             continue
#                         progreso.update(hasta - desde)
#                         desde = hasta
#                         del entrada, salida
#             return proba
#     except BaseException as error:
#         # No dejar temporales GPU en los frames de una ejecución fallida/interrumpida.
#         traceback.clear_frames(error.__traceback__)
#         raise
#     finally:
#         if estimador is not None:
#             estimador.model_kv_cache_ = None
#             estimador.ensemble_generator_ = None
#         modelo.clear_cache()
#         modelo.train(modo_tfm)
#         if adaptador is not None:
#             adaptador.train(modo_adaptador)
#         del contexto, estimador
#         gc.collect()
#         torch.cuda.empty_cache()


In [13]:
from tabicl._model.kv_cache import KVCache


class EntradaPreparada:
    def transform(self, X):
        return np.asarray(X)


class VistaPreparada:
    def __init__(self, dimension, metodo):
        self.metodo = metodo
        self.feature_shuffles_ = {metodo: [list(range(dimension))]}
        self.class_shuffles_ = {metodo: [[0, 1]]}

    def transform(self, X, mode="test"):
        return OrderedDict([(self.metodo, (np.asarray(X)[None],))])


class CacheICLRAM(KVCache):
    def to(self, device, dtype=None):
        # Evitar que el gestor suba todas las capas juntas a GPU.
        return CacheICLRAM(kv={
            i: entrada.to("cpu", dtype=dtype)
            for i, entrada in self.kv.items()
        })


def cargar_kv(modulo, entrada, argumentos):
    torch.cuda.synchronize(DEVICE)
    cache = argumentos.get("cached_kv")
    if cache is not None:
        # Copia temporal de una sola capa; el original permanece en RAM.
        return entrada, {**argumentos, "cached_kv": cache.to(DEVICE)}


def compactar_kv(modulo, entrada, salida):
    if isinstance(salida, tuple):
        # Conservar FP32 y copiar directamente a RAM, sin clonar en GPU.
        kv = [
            t.to(device="cpu", memory_format=torch.contiguous_format)
            for t in salida[1:]
        ]
        torch.cuda.synchronize(DEVICE)
        capa = len(modelo._cache.icl_cache.kv) + 1
        total = len(modelo.icl_predictor.tf_icl.blocks)
        print(f"Caché ICL: {capa}/{total} capas en RAM.", flush=True)
        return (salida[0], *kv)

    torch.cuda.synchronize(DEVICE)


def preparar_cache(estimador, X, y, metodo):
    estimador._resolve_device()
    estimador._build_inference_config()
    estimador.n_samples_in_, estimador.n_features_in_ = X.shape
    estimador.y_encoder_ = LabelEncoder().fit(y)
    estimador.classes_ = estimador.y_encoder_.classes_
    estimador.n_classes_ = len(estimador.classes_)
    estimador.X_encoder_ = EntradaPreparada()
    estimador.ensemble_generator_ = VistaPreparada(X.shape[1], metodo)
    estimador.cache_mode_ = estimador.kv_cache

    entrada = torch.from_numpy(X[None]).float().to(DEVICE)
    etiquetas = torch.tensor(y[None], dtype=torch.float32, device=DEVICE)

    try:
        with ExitStack() as pila:
            pila.callback(
                modelo.row_interactor.register_forward_hook(
                    lambda *args: torch.cuda.synchronize(DEVICE)
                ).remove
            )
            modelo.forward_with_cache(
                X_train=entrada,
                y_train=etiquetas,
                store_cache=True,
                cache_mode=estimador.cache_mode_,
                inference_config=estimador.inference_config_,
            )
    finally:
        del entrada, etiquetas

    modelo._cache.icl_cache = CacheICLRAM(
        kv=modelo._cache.icl_cache.kv
    )
    estimador.model_kv_cache_ = OrderedDict([(metodo, modelo._cache)])
    modelo.clear_cache()

In [14]:
import traceback
from tqdm.auto import tqdm


def adaptar_numpy(X, adaptador):
    if adaptador is None:
        return X

    with torch.no_grad():
        entrada = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
        salida = adaptador(entrada).float().cpu().numpy()

    assert salida.shape == X.shape and np.isfinite(salida).all()
    return salida


@torch.no_grad()
def predecir(X_contexto, y_contexto, X_consultas, adaptador, parametros):
    e = parametros["ejecucion"]
    metodo = parametros["fijos"]["preprocesamiento"]["norm_methods"]
    tamanos = e["tamanos_consulta_oom"]

    modo_tfm = modelo.training
    modo_adaptador = adaptador.training if adaptador is not None else None
    modelo.eval()

    if adaptador is not None:
        adaptador.eval()
        adaptador.zero_grad(set_to_none=True)

    modelo.clear_cache()
    modelo.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()

    contexto = estimador = None
    raiz = Path(e["offload_raiz"]).expanduser()
    raiz.mkdir(parents=True, exist_ok=True)

    try:
        with tempfile.TemporaryDirectory(dir=raiz) as temporal, ExitStack() as pila:
            # Activos solo durante esta predicción; se retiran también ante errores.
            for bloque in modelo.icl_predictor.tf_icl.blocks:
                pila.callback(
                    bloque.register_forward_pre_hook(
                        cargar_kv, with_kwargs=True
                    ).remove
                )
                pila.callback(
                    bloque.register_forward_hook(compactar_kv).remove
                )

            comun = {
                "use_amp": False,
                "use_fa3": e["use_fa3"],
                "safety_factor": e["safety_factor"],
                "use_async": e["use_async"],
            }
            inferencia = {
                nombre: dict(comun)
                for nombre in ("COL_CONFIG", "ROW_CONFIG", "ICL_CONFIG")
            }
            inferencia["COL_CONFIG"].update(
                cpu_safety_factor=e["cpu_safety_factor"],
                max_pinned_memory_mb=e["max_pinned_memory_mb"],
                disk_offload_dir=temporal,
            )

            estimador = trainer._build_inner_estimator(
                modelo,
                parametros["fijos"]["n_estimators_inference"],
                DEVICE,
            )
            estimador.set_params(
                kv_cache=e["kv_cache"],
                batch_size=e["batch_size_api"],
                use_amp=False,
                use_fa3=e["use_fa3"],
                disk_offload_dir=temporal,
                inference_config=inferencia,
            )

            with torch.autocast(device_type=DEVICE.type, enabled=False):
                contexto = X_contexto

                if adaptador is not None:
                    contexto = np.empty_like(X_contexto)
                    for desde in tqdm(
                        range(0, len(contexto), tamanos[0]),
                        desc="Adaptando contexto",
                        unit="bloque",
                    ):
                        hasta = min(desde + tamanos[0], len(contexto))
                        contexto[desde:hasta] = adaptar_numpy(
                            X_contexto[desde:hasta], adaptador
                        )

                print(
                    f"Creando caché del contexto: {len(contexto):,} filas...",
                    flush=True,
                )
                inicio = time.perf_counter()
                preparar_cache(estimador, contexto, y_contexto, metodo)
                torch.cuda.synchronize(DEVICE)
                print(
                    f"Caché lista en {(time.perf_counter()-inicio)/60:.2f} min.",
                    flush=True,
                )

                contexto = None
                proba = np.empty(len(X_consultas), dtype=np.float64)
                desde = indice_tamano = 0

                with tqdm(
                    total=len(X_consultas),
                    desc="Prediciendo",
                    unit="filas",
                ) as progreso:
                    while desde < len(X_consultas):
                        hasta = min(
                            desde + tamanos[indice_tamano],
                            len(X_consultas),
                        )
                        try:
                            entrada = adaptar_numpy(
                                X_consultas[desde:hasta], adaptador
                            )
                            salida = estimador.predict_proba(entrada)
                            torch.cuda.synchronize(DEVICE)

                            if not np.isfinite(salida).all():
                                raise FloatingPointError(
                                    f"Probabilidades no finitas en filas "
                                    f"{desde}:{hasta}."
                                )

                            proba[desde:hasta] = salida[:, 1]

                        except torch.OutOfMemoryError as error:
                            traceback.clear_frames(error.__traceback__)
                            modelo.clear_cache()
                            gc.collect()
                            torch.cuda.empty_cache()

                            indice_tamano += 1
                            if indice_tamano == len(tamanos):
                                raise

                            tqdm.write(
                                f"OOM de consulta; reintentando con "
                                f"{tamanos[indice_tamano]} filas."
                            )
                            continue

                        progreso.update(hasta - desde)
                        desde = hasta
                        del entrada, salida

            return proba

    except BaseException as error:
        traceback.clear_frames(error.__traceback__)
        raise

    finally:
        if estimador is not None:
            estimador.model_kv_cache_ = None
            estimador.ensemble_generator_ = None

        modelo.clear_cache()
        modelo.train(modo_tfm)

        if adaptador is not None:
            adaptador.train(modo_adaptador)

        del contexto, estimador
        gc.collect()
        torch.cuda.empty_cache()

### 5.3 Entrenamiento de una combinación


In [15]:
def entrenar(X, y, parametros, semilla, epochs, evaluar=None):
    f = parametros["fijos"]
    hp = parametros["hp"]
    opt = f["optimizacion"]
    assert opt["scheduler"] == "coseno"
    fijar_semilla(semilla)
    adaptador = AdaptadorCross(X.shape[1], hp, f["adaptador"]).to(DEVICE)
    grupos = [
        {"params": [p for n, p in adaptador.named_parameters() if n != "alpha" and p.ndim >= 2]},
        {"params": [p for n, p in adaptador.named_parameters() if n != "alpha" and p.ndim < 2],
         "weight_decay": opt["weight_decay_otros"]},
        {"params": [adaptador.alpha], "lr": hp["lr_maximo"] * hp["factor_lr_compuerta"],
         "weight_decay": opt["weight_decay_compuerta"]},
    ]
    optimizador = getattr(torch.optim, opt["optimizador"])(grupos, lr=hp["lr_maximo"])
    _, scaler, amp_context = trainer._make_amp(DEVICE)
    n_clases = len(np.unique(y))
    pasos_epoca = math.ceil(len(y) / f["filas_por_bloque"])
    pasos_totales = f["epochs"] * pasos_epoca
    pasos_warmup = opt["warmup_epochs"] * pasos_epoca

    def factor_lr(paso):
        if paso < pasos_warmup:
            return paso / max(1, pasos_warmup)
        progreso = (paso - pasos_warmup) / max(1, pasos_totales - pasos_warmup)
        return max(0.0, (1 + math.cos(math.pi * progreso)) / 2)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)
    for epoca in range(1, epochs + 1):
        fijar_semilla(semilla + epoca)
        trainer._set_training_mode(True)
        adaptador.train()
        modelo.clear_cache()
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)
        inicio = time.perf_counter()
        orden = np.random.default_rng(np.random.SeedSequence([semilla, epoca])).permutation(len(y))
        suma_loss = vistas = updates = 0
        for numero, posiciones in enumerate(np.array_split(orden, pasos_epoca), 1):
            etiquetas = y[posiciones]
            contexto, consultas = _split_ctx_query(
                etiquetas, query_size=max(2, int(len(posiciones) * trainer.finetune_ctx_query_ratio)),
                seed=semilla + epoca + (numero - 1) * 7919, stratify=True,
            )
            indices = np.concatenate([contexto, consultas])
            batch = MetaBatch(
                X=torch.from_numpy(np.ascontiguousarray(X[posiciones[indices]])).unsqueeze(0),
                y_train=torch.from_numpy(etiquetas[contexto].astype(np.float32)).unsqueeze(0),
                y_query=torch.from_numpy(etiquetas[consultas]).unsqueeze(0), train_size=len(contexto),
            )
            assert not trainer._task_skip_batch(batch), "Consulta con clase ausente del contexto."
            batch = move_meta_batch(batch, DEVICE)
            optimizador.zero_grad(set_to_none=True)
            with amp_context():
                logits = modelo(adaptador(batch.X), batch.y_train)
                loss = torch.nn.functional.cross_entropy(
                    logits[..., :n_clases].reshape(-1, n_clases), batch.y_query.reshape(-1)
                )
            if not torch.isfinite(loss).item():
                raise FloatingPointError(f"Loss no finita en época {epoca}, bloque {numero}.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(adaptador.parameters(), trainer.grad_clip)
            escala = scaler.get_scale()
            scaler.step(optimizador)
            scaler.update()
            scheduler.step()
            updates += int(scaler.get_scale() >= escala)
            vistas += len(posiciones)
            suma_loss += loss.item()
            del batch, logits, loss
            if numero == 1 or numero % parametros["ejecucion"]["log_cada_bloques"] == 0 or numero == pasos_epoca:
                print(f"Época {epoca}/{epochs} | bloque {numero}/{pasos_epoca} | "
                      f"loss={suma_loss/numero:.4f} | LR={optimizador.param_groups[0]['lr']:.2e}", flush=True)
        assert vistas == len(y) and updates > 0
        torch.cuda.synchronize(DEVICE)
        print(f"Época {epoca}: {(time.perf_counter()-inicio)/60:.2f} min | "
              f"updates={updates}/{pasos_epoca} | "
              f"pico VRAM={torch.cuda.max_memory_reserved(DEVICE)/2**30:.2f} GiB", flush=True)
        optimizador.zero_grad(set_to_none=True)
        modelo.clear_cache()
        gc.collect()
        torch.cuda.empty_cache()
        if evaluar is not None and epoca in f["evaluar_epochs"]:
            evaluar(epoca, adaptador)
    return adaptador


## 6. Búsqueda de hiperparámetros
### 6.1 Grid y validación


In [16]:
if PARAM["fijos"]["correr_busqueda"]:
    assert PARAM["fijos"]["metrica_seleccion"] == "auc"
    if ARCHIVO_LOG.exists():
        ARCHIVO_LOG.rename(ARCHIVO_LOG.with_name(f"Grid_log_{datetime.now():%Y%m%d_%H%M%S}.txt"))
    combinaciones = [dict(zip(hs, valores)) for valores in product(*hs.values())]
    print("Combinaciones:", len(combinaciones), flush=True)
    metrica_0 = None
    if PARAM["fijos"]["evaluar_epoch_0"]:
        proba = predecir(X_train, y_train, X_valid, None, PARAM)
        metrica_0 = roc_auc_score(y_valid, proba)
        del proba

    for iteracion, hp in enumerate(combinaciones, 1):
        parametros = {**deepcopy(PARAM), "hp": hp}
        print(f"\nCombinación {iteracion}/{len(combinaciones)}: {hp}", flush=True)
        if metrica_0 is not None:
            loguear(iteracion, 0, metrica_0, parametros)

        def evaluar_validacion(epoca, adaptador):
            proba = predecir(X_train, y_train, X_valid, adaptador, parametros)
            loguear(iteracion, epoca, roc_auc_score(y_valid, proba), parametros)

        adaptador = entrenar(
            X_train, y_train, parametros, SEED, parametros["fijos"]["epochs"], evaluar_validacion
        )
        del adaptador
        gc.collect()
        torch.cuda.empty_cache()


### 6.2 Mejores hiperparámetros y época


In [17]:
tb_grid = pd.read_csv(ARCHIVO_LOG, sep="\t")
# Mayor AUC; ante empate, preferir menor época y luego menor número de combinación.
tb_grid = tb_grid.sort_values(["metrica", "epoch", "iter"], ascending=[False, True, True])
mejor = tb_grid.iloc[0]
param_final = json.loads(mejor["parametros"])
epoca_final = int(mejor["epoch"])
print("Mejores HP:", param_final["hp"])
print("Época:", epoca_final, "| AUC validación:", mejor["metrica"])
display(tb_grid.pivot(index="iter", columns="epoch", values="metrica"))


Mejores HP: {'lr_maximo': 0.0001, 'factor_lr_compuerta': 3.0, 'compuerta_inicial': 0.02, 'batchnorm': True}
Época: 10 | AUC validación: 0.9417813378535448


epoch,0,10,20,30
iter,,,,
1,0.941654,0.941781,0.938884,0.936734
2,0.941654,0.940391,0.940833,0.941609


## 7. Producción
Solo si `produccion.correr_prod` está activado. Se entrena desde cero sobre todos los
meses de `train_final`, con undersampling y los HP/época elegidos. Como en las referencias,
cada semilla de producción determina tanto el muestreo como el entrenamiento.
El scheduler mantiene el horizonte original de la búsqueda.


In [18]:
def generar_semillas(cantidad, semilla):
    # Primos de seis cifras sorteados a partir de la primigenia, como en la referencia.
    rng = np.random.default_rng(semilla)
    semillas = []
    while len(semillas) < cantidad:
        candidato = int(rng.integers(100000, 1000000))
        primo = all(candidato % d for d in range(2, math.isqrt(candidato) + 1))
        if primo and candidato not in semillas:
            semillas.append(candidato)
    return semillas


def entrenar_y_predecir(semilla):
    training = param_final["training"]
    indices = metadatos.index[metadatos["foto_mes"].isin(training["train_final"])].to_numpy()
    indices = seleccionar_training(indices, semilla, training)
    futuro = metadatos.index[metadatos["foto_mes"].isin(training["future"])].to_numpy()
    X, y, X_future = preparar_datos(indices, futuro, semilla, param_final)
    adaptador = entrenar(X, y, param_final, semilla, epoca_final) if epoca_final else None
    try:
        return predecir(X, y, X_future, adaptador, param_final)
    finally:
        del X, y, X_future, adaptador
        gc.collect()
        torch.cuda.empty_cache()


In [19]:
# prob_semillas = None
# if config["produccion"]["correr_prod"]:
#     # La búsqueda ya terminó; liberar sus matrices procesadas, conservar el baseline.
#     del X_train, y_train, X_valid
#     gc.collect()
#     semillas_finales = generar_semillas(PARAM["training"]["semillas_finales"], SEED)
#     idx_future = metadatos.index[metadatos["foto_mes"].isin(param_final["training"]["future"])].to_numpy()
#     meta_future = metadatos.loc[idx_future].reset_index(drop=True)
#     prob_semillas = pd.DataFrame({"numero_de_cliente": meta_future["numero_de_cliente"]})
#     for numero, semilla in enumerate(semillas_finales, 1):
#         prob_semillas[f"S_{semilla}"] = entrenar_y_predecir(semilla)
#         print(f"Producción: {numero}/{len(semillas_finales)} semillas", flush=True)
#     prob_semillas.to_csv(EXP_DIR / "prob_202106_por_semilla.csv", index=False)
#     prob_final = prob_semillas.filter(like="S_").mean(axis=1).to_numpy()


In [20]:

prob_semillas = None
if config["produccion"]["correr_prod"]:
    for nombre in ("X_train", "y_train", "X_valid"):
        globals().pop(nombre, None)
    gc.collect()

    cantidad = int(PARAM["training"]["semillas_finales"])
    assert cantidad > 0

    idx_future = metadatos.index[
        metadatos["foto_mes"].isin(param_final["training"]["future"])
    ].to_numpy()
    meta_future = metadatos.loc[idx_future].reset_index(drop=True)

    prob_semillas = pd.DataFrame({
        "numero_de_cliente": meta_future["numero_de_cliente"]
    })
    archivo = EXP_DIR / "prob_202106_por_semilla.csv"

    semillas_finales = []
    intentos = 0

    while len(semillas_finales) < cantidad:
        intentos += 1
        semilla = generar_semillas(intentos, SEED)[-1]

        print(
            f"Intento {intentos} | semilla={semilla} | "
            f"completas={len(semillas_finales)}/{cantidad}",
            flush=True
        )

        try:
            probabilidades = entrenar_y_predecir(semilla)

            if (
                probabilidades.shape != (len(meta_future),)
                or not np.isfinite(probabilidades).all()
            ):
                raise FloatingPointError(
                    "Predicciones incompletas o no finitas."
                )

        except Exception as error:
            print(
                f"Falló semilla {semilla}: {type(error).__name__}: {error}. "
                "Se continúa con otra semilla.",
                flush=True
            )
            traceback.clear_frames(error.__traceback__)
            modelo.clear_cache()
            modelo.zero_grad(set_to_none=True)
            gc.collect()
            torch.cuda.empty_cache()
            continue

        prob_semillas[f"S_{semilla}"] = probabilidades
        prob_semillas.to_csv(archivo, index=False)
        semillas_finales.append(semilla)
        del probabilidades

        print(
            f"Predicciones guardadas: {len(semillas_finales)}/{cantidad} | "
            f"semilla={semilla}",
            flush=True
        )

    prob_final = prob_semillas.filter(like="S_").mean(axis=1).to_numpy()

Intento 1 | semilla=591233 | completas=0/10
Época 1/10 | bloque 1/852 | loss=0.2288 | LR=1.17e-07


/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /opt/conda/conda-bld/pytorch_1729647348947/work/aten/src/ATen/native/transformers/cuda/attention_backward.cu:102.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/home/marco/anaconda3/envs/cmeyf-cuda13/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra

Época 1/10 | bloque 20/852 | loss=0.1973 | LR=2.35e-06
Época 1/10 | bloque 40/852 | loss=0.1960 | LR=4.69e-06
Época 1/10 | bloque 60/852 | loss=0.1960 | LR=7.04e-06
Época 1/10 | bloque 80/852 | loss=0.1998 | LR=9.39e-06
Época 1/10 | bloque 100/852 | loss=0.1955 | LR=1.17e-05
Época 1/10 | bloque 120/852 | loss=0.1913 | LR=1.41e-05
Época 1/10 | bloque 140/852 | loss=0.1909 | LR=1.64e-05
Época 1/10 | bloque 160/852 | loss=0.1916 | LR=1.88e-05
Época 1/10 | bloque 180/852 | loss=0.1924 | LR=2.11e-05
Época 1/10 | bloque 200/852 | loss=0.1938 | LR=2.35e-05
Época 1/10 | bloque 220/852 | loss=0.1932 | LR=2.58e-05
Época 1/10 | bloque 240/852 | loss=0.1921 | LR=2.82e-05
Época 1/10 | bloque 260/852 | loss=0.1916 | LR=3.05e-05
Época 1/10 | bloque 280/852 | loss=0.1908 | LR=3.29e-05
Época 1/10 | bloque 300/852 | loss=0.1896 | LR=3.52e-05
Época 1/10 | bloque 320/852 | loss=0.1903 | LR=3.76e-05
Época 1/10 | bloque 340/852 | loss=0.1902 | LR=3.99e-05
Época 1/10 | bloque 360/852 | loss=0.1895 | LR=4.23e

Adaptando contexto:   0%|          | 0/852 [00:00<?, ?bloque/s]

Creando caché del contexto: 435,955 filas...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 21.84 min.


Prediciendo:   0%|          | 0/164114 [00:00<?, ?filas/s]

... (59 outputs left out)

Adaptando contexto:   0%|          | 0/855 [00:00<?, ?bloque/s]

Creando caché del contexto: 437,536 filas...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 21.96 min.


Prediciendo:   0%|          | 0/164114 [00:00<?, ?filas/s]

Predicciones guardadas: 10/10 | semilla=846529


### 7.1 Ganancia en el mes futuro


In [29]:
if config["produccion"]["correr_prod"]:
    auc_test = roc_auc_score(meta_future["clase01"], prob_final)
    orden = np.argsort(-prob_final, kind="stable")
    # Misma ganancia bancaria que las referencias: beneficio bruto menos costo.
    ganancia = np.where(meta_future["clase_ternaria"].to_numpy()[orden] == "BAJA+2", 1072500, -27500)
    curva = pd.DataFrame({"envios": np.arange(1, len(orden) + 1), "prob": prob_final[orden],
                          "ganancia_acumulada": np.cumsum(ganancia)})
    envios_max = int(curva.loc[curva["ganancia_acumulada"].idxmax(), "envios"])
    
    cortes = sorted(set(param_final["training"]["cortes"] + [envios_max]))
    tabla_ganancia = curva.loc[curva["envios"].isin(cortes), ["envios", "ganancia_acumulada"]]
    tabla_ganancia.to_csv(EXP_DIR / "ganancia_por_envios.csv", index=False)
    curva.to_csv(EXP_DIR / "curva_ganancia.csv", index=False)
    pd.DataFrame([{"auc_202106": auc_test, "epoch": epoca_final, **param_final["hp"],
                   "ganancia_maxima": curva["ganancia_acumulada"].max(), "envios_maximo": envios_max}
                 ]).to_csv(EXP_DIR / "resumen_produccion.csv", index=False)
    print("AUC futuro:", auc_test, "| ganancia máxima:", curva["ganancia_acumulada"].max(), "| envíos:", envios_max)
    display(tabla_ganancia)
    go.Figure(go.Scatter(x=curva["envios"], y=curva["ganancia_acumulada"], mode="lines")).update_layout(
        title="Ganancia acumulada en el mes futuro", xaxis_title="Envíos", yaxis_title="Ganancia", xaxis_range=[0, 16000],yaxis_range=[0, 500000000]
    ).show()


AUC futuro: 0.9198604318559523 | ganancia máxima: 427240000 | envíos: 10824


,envios,ganancia_acumulada
8999,9000,408100000
9499,9500,413050000
9999,10000,416900000
10499,10500,419650000
10823,10824,427240000
10999,11000,422400000
11499,11500,419650000
11999,12000,423500000
12499,12500,418550000
12999,13000,421300000


## 8. Runner de predicciones
Solo si `predicciones.activar_runner` está activado. Recupera los HP y la época del log,
genera las probabilidades por semilla y guarda el avance después de cada modelo, como en XGBoost.


In [22]:
if config["predicciones"]["activar_runner"]:
    # También funciona si producción está desactivada.
    for nombre in ("X_train", "y_train", "X_valid"):
        globals().pop(nombre, None)
    gc.collect()
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    archivo = PRED_DIR / config["predicciones"]["predicciones_filename"]
    semillas_runner = generar_semillas(config["predicciones"]["semillas_runner"], SEED)
    idx_future = metadatos.index[metadatos["foto_mes"].isin(param_final["training"]["future"])].to_numpy()
    ids = metadatos.loc[idx_future, "numero_de_cliente"].to_numpy()
    predicciones = pd.read_csv(archivo) if archivo.exists() else pd.DataFrame({"numero_de_cliente": ids})
    assert np.array_equal(predicciones["numero_de_cliente"].to_numpy(), ids)
    for semilla in semillas_runner:
        columna = f"S_{semilla}"
        if columna not in predicciones:
            if prob_semillas is not None and columna in prob_semillas:
                predicciones[columna] = prob_semillas[columna].to_numpy()
            else:
                predicciones[columna] = entrenar_y_predecir(semilla)
            predicciones.to_csv(archivo, index=False)
        completas = sum(f"S_{s}" in predicciones for s in semillas_runner)
        print(f"Predicciones guardadas: {completas}/{len(semillas_runner)} | semilla={semilla}", flush=True)


# 9 Modelo sin fine tuning para comparar 


In [30]:
# 9. Modelo sin fine-tuning

cantidad_sin_ft = 3
archivo_sin_ft = EXP_DIR / "modelo_sin_ft.csv"
training_sin_ft = param_final["training"]
assert training_sin_ft["future"] == [202106]
assert training_sin_ft["undersampling"] == 0.1

semillas_sin_ft = generar_semillas(cantidad_sin_ft, SEED)
historico_sin_ft = metadatos.index[
    metadatos["foto_mes"].isin(training_sin_ft["train_final"])
].to_numpy()
futuro_sin_ft = metadatos.index[
    metadatos["foto_mes"].isin(training_sin_ft["future"])
].to_numpy()
clases_sin_ft = metadatos.loc[futuro_sin_ft, "clase_ternaria"].to_numpy()
probas_sin_ft = np.empty((len(futuro_sin_ft), cantidad_sin_ft))

for i, semilla in enumerate(semillas_sin_ft):
    fijar_semilla(semilla)
    indices = seleccionar_training(historico_sin_ft, semilla, training_sin_ft)
    X, y, X_future = preparar_datos(indices, futuro_sin_ft, semilla, param_final)
    print(f"Sin FT: {i+1}/{cantidad_sin_ft} | semilla={semilla}", flush=True)
    try:
        # Sin entrenamiento ni adaptador; misma inferencia FP32 de 5.2.
        probas_sin_ft[:, i] = predecir(X, y, X_future, None, param_final)
    finally:
        del X, y, X_future
        gc.collect()
        torch.cuda.empty_cache()

resultados_sin_ft = []
evaluaciones_sin_ft = [
    (str(s), probas_sin_ft[:, i]) for i, s in enumerate(semillas_sin_ft)
] + [("promedio", probas_sin_ft.mean(axis=1))]

for semilla, proba in evaluaciones_sin_ft:
    if not np.isfinite(proba).all():
        raise FloatingPointError(f"Probabilidades no finitas: {semilla}.")
    orden = np.argsort(-proba, kind="stable")
    curva = np.cumsum(
        np.where(clases_sin_ft[orden] == "BAJA+2", 1072500, -27500)
    )
    for corte in training_sin_ft["cortes"]:
        resultados_sin_ft.append({
            "modelo": "modelo_sin_ft",
            "semilla": semilla,
            "envios": corte,
            "ganancia": curva[corte - 1],
        })

resultados_sin_ft = pd.DataFrame(resultados_sin_ft)
resultados_sin_ft.to_csv(archivo_sin_ft, index=False)
display(resultados_sin_ft)
print("Guardado:", archivo_sin_ft)

Sin FT: 1/3 | semilla=591233
Creando caché del contexto: 435,955 filas...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 22.91 min.


Prediciendo:   0%|          | 0/164114 [00:00<?, ?filas/s]

Sin FT: 2/3 | semilla=116471
Creando caché del contexto: 436,210 filas...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 22.36 min.


Prediciendo:   0%|          | 0/164114 [00:00<?, ?filas/s]

Sin FT: 3/3 | semilla=264371
Creando caché del contexto: 437,080 filas...
Caché ICL: 1/12 capas en RAM.
Caché ICL: 2/12 capas en RAM.
Caché ICL: 3/12 capas en RAM.
Caché ICL: 4/12 capas en RAM.
Caché ICL: 5/12 capas en RAM.
Caché ICL: 6/12 capas en RAM.
Caché ICL: 7/12 capas en RAM.
Caché ICL: 8/12 capas en RAM.
Caché ICL: 9/12 capas en RAM.
Caché ICL: 10/12 capas en RAM.
Caché ICL: 11/12 capas en RAM.
Caché ICL: 12/12 capas en RAM.
Caché lista en 22.80 min.


Prediciendo:   0%|          | 0/164114 [00:00<?, ?filas/s]

,modelo,semilla,envios,ganancia
0,modelo_sin_ft,591233,9000,392700000
1,modelo_sin_ft,591233,9500,402050000
2,modelo_sin_ft,591233,10000,412500000
3,modelo_sin_ft,591233,10500,418550000
4,modelo_sin_ft,591233,11000,419100000
5,modelo_sin_ft,591233,11500,415250000
6,modelo_sin_ft,591233,12000,418000000
7,modelo_sin_ft,591233,12500,420750000
8,modelo_sin_ft,591233,13000,420200000
9,modelo_sin_ft,591233,13500,418550000


Guardado: /home/marco/code/DMEyF/exp/TabICL_Retouche/modelo_sin_ft.csv
